# Lab | LLMs through an API
## A support assistant for the bank

Over the last two days you routed the bank's customer messages with semantic search and with a fine-tuned DistilBERT. Both needed hundreds of labelled messages. The product team now asks a different question: **could a large language model do the job with no training at all, just a good prompt?** And while it is at it, could it extract the details of each message and draft the reply?

You will build that assistant with the OpenAI API on 60 real messages from the ten intents of yesterday's lab, measure its accuracy against yesterday's models, and keep track of what every step costs.

**The data** is a sample of the Banking77 dataset: `data/messages.csv` has the 60 test messages (6 per intent), and `data/examples.csv` has 20 other labelled messages (2 per intent) that you can use as examples in prompts.

**This lab practises** calling an LLM through an API. You will use, from today's **LLMs through an API** lesson: messages and roles, the system prompt, `temperature`, the `usage` field and the cost per token, zero-shot and few-shot prompts, structured outputs with Pydantic, and conversation history.

**By the end, you can:**

1. Implement an LLM-based classifier and extractor with prompts and structured outputs, and measure its accuracy and cost.
2. Explain when a prompt is enough and when a trained model is the better choice.

Each challenge ends with a **checkpoint** that tells you what you should see. The model's answers change a little from run to run, so your numbers can differ by a few points. **The whole lab costs less than 5 US cents** with the model below; the last cell shows your total.

## Setup

You need `pip install openai python-dotenv pydantic`. Create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...` (the key Ironhack gave you). The repository's `.gitignore` already lists `.env`, so git will not see it. **Never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

In [1]:
pip install openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []                               # one row per API call, so you can add up the cost at the end

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

Key found: True


In [3]:
messages = pd.read_csv("../data/messages.csv")
examples = pd.read_csv("../data/examples.csv")
intents = sorted(messages["category"].unique())
print(messages.shape, examples.shape)
print(intents)

(60, 2) (20, 2)
['card_arrival', 'card_delivery_estimate', 'card_not_working', 'change_pin', 'compromised_card', 'exchange_rate', 'lost_or_stolen_card', 'pin_blocked', 'top_up_failed', 'top_up_reverted']


**Checkpoint:** `Key found: True`, then `(60, 2) (20, 2)` and the ten intents of yesterday's lab, from `card_arrival` to `top_up_reverted`.

## Challenge 1: a first call, and what it costs

1. Write a function `cost(input_tokens, output_tokens)` that returns the cost in US dollars with the prices in `PRICES[MODEL]`.
2. Send the first customer message (`messages["text"][0]`) to the model with a system prompt that makes it a polite bank support assistant answering in at most two sentences. Wrap the call in `track(..., "first call")`.
3. Print the reply, the number of input and output tokens, and the cost of the call.

In [5]:
# 1. Cost of a call
def cost(input_tokens, output_tokens, model=MODEL):
    price_in, price_out = PRICES[model]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

# 2. First call
system_prompt = 'You are a polite support assistant of a bank. You answer at most two sentences.'
response = track(
    client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': messages['text'][0]}
        ],
    ),
    'first call'
)

# 3. Reply, tokens and cost
reply = response.choices[0].message.content
tokens_in = response.usage.prompt_tokens
tokens_out = response.usage.completion_tokens
print('Reply:', reply)
print(f'Input Tokens: {tokens_in} | Output Tokens: {tokens_out}')
print(f'Cost: ${cost(tokens_in, tokens_out)}')

Reply: I’m sorry this is happening—yes, we can help you freeze your card immediately. Please tell me which card type/bank name you have and, if available, the last 4 digits (or your customer ID), and we’ll guide you through the fastest way to lock it.
Input Tokens: 56 | Output Tokens: 61
Cost: $8.745e-05


**Checkpoint:** a short, polite reply, about 60 input and 50 output tokens, and a cost below a hundredth of a cent (about $0.00008). Read the reply: with no rules in the system prompt, the model may ask for details a bank should never ask for in a chat (in the reference run, it asked for the last four digits of the card and the billing postcode). Challenge 5 fixes that with constraints.

## Challenge 2: zero-shot intent classification

1. Write a system prompt that asks the model to classify a customer message into exactly one of the ten intents, lists the intents (you can join the list `intents` into the prompt), and asks for **the intent name only**.
2. Write `classify(text)`: one call with your system prompt and the message, `temperature=0`, tracked as `"zero-shot"`, returning the reply with spaces removed (`.strip()`).
3. Classify the 60 messages, store the predictions in a column `zero_shot`, and compute the accuracy.
4. Count the predictions that are **not** one of the ten intents.

In [10]:
# 1. System prompt
system_prompt_classify = (
    "You classify customer messages sent to a bank's support team. "
    "Classify the customer messages into exactly one of these intents:\n"
    +"\n".join(intents)
    +"\n\n Reply with the intent name only exactly as written above."
)
print(system_prompt_classify)

# 2. Classifier
def classify(text):
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[
                {'role': 'system', 'content': system_prompt_classify},
                {'role': 'user', 'content': text}
            ]
        ),
        'zero-shot'
    )
    return response.choices[0].message.content.strip()

# 3. Predictions and accuracy
messages['zero_shot'] = messages['text'].apply(classify)
zero_shot_acc = (messages['zero_shot'] == messages['category']).mean()
print(f'Zero-shot accuracy: {zero_shot_acc:.3f}')

# 4. Invalid answers
invalid = ~messages['zero_shot'].isin(intents)
print('Predictions that are not one of the 10 intents:', invalid.sum())

You classify customer messages sent to a bank's support team. Classify the customer messages into exactly one of these intents:
card_arrival
card_delivery_estimate
card_not_working
change_pin
compromised_card
exchange_rate
lost_or_stolen_card
pin_blocked
top_up_failed
top_up_reverted

 Reply with the intent name only exactly as written above.
Zero-shot accuracy: 0.883
Predictions that are not one of the 10 intents: 0


**Checkpoint:** an accuracy between about 0.8 and 0.9 (0.88 in the reference solution), and no answers outside the ten intents, if your prompt asks for the name only.

In [11]:
# 5. Show the messages that the zero-shot classifier got wrong, with the true and the predicted intent
wrong_messages = messages[messages['zero_shot'] != messages['category']]
display(wrong_messages[['text', 'category', 'zero_shot']])

,text,category,zero_shot
5,What is the tracking number for my card that was mailed?,card_arrival,card_delivery_estimate
10,How do I track my card?,card_arrival,card_delivery_estimate
28,"I have been waiting longer than expected for my bank card, could you provide information on when...",card_arrival,card_delivery_estimate
30,How do I unblock my card using the app?,card_not_working,pin_blocked
36,How do I freeze my card using the app?,compromised_card,lost_or_stolen_card
52,When will my card arrive?,card_delivery_estimate,card_arrival
56,How long should my new card take to arrive?,card_arrival,card_delivery_estimate


**Question:** look at the mistakes. Which intents does the model confuse, and is it the same pattern as yesterday's models?

The model confuses the intents that are closely related, even a person would be unsure which one is more applicable. It's quite similar to yesterday's pattern of confusing the intents.

## Challenge 3: few-shot classification

The intent names do not tell the model where the bank draws the line between, say, `card_arrival` and `card_delivery_estimate`. Examples can.

1. Build `few_shot_messages`: your system prompt, followed by the 20 rows of `examples` as pairs of messages, `{"role": "user", "content": text}` then `{"role": "assistant", "content": category}`.
2. Write `classify_few_shot(text)`, like `classify` but sending `few_shot_messages` plus the new message, tracked as `"few-shot"`.
3. Classify the 60 messages into a column `few_shot` and compute the accuracy.
4. From `usage_log`, compute the average number of input tokens per call for zero-shot and for few-shot, and the cost of classifying **1,000** messages with each.

In [12]:
# 1. Few-shot messages
few_shot_messages = [{'role': 'system', 'content': system_prompt_classify}]
for row in examples.itertuples():
    few_shot_messages.append({'role': 'user', 'content': row.text})
    few_shot_messages.append({'role': 'assistant', 'content': row.category})

# 2. Classifier
def classify_few_shot(text):
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=few_shot_messages + [{'role': 'user', 'content': text}]
        ),
        'few-shot'
    )
    return response.choices[0].message.content.strip()

# 3. Predictions and accuracy
messages["few_shot"] = messages["text"].apply(classify_few_shot)
few_shot_acc = (messages["few_shot"] == messages["category"]).mean()
print(f"Zero-shot accuracy: {zero_shot_acc:.2f}")
print(f"Few-shot accuracy:  {few_shot_acc:.2f}")
print("Invalid answers:", (~messages["few_shot"].isin(intents)).sum())

# 4. Tokens per call and cost per 1,000 messages
log = pd.DataFrame(usage_log)
per_call = log[log["step"].isin(["zero-shot", "few-shot"])].groupby("step")[["input", "output"]].mean()
per_call["cost per 1,000 ($)"] = [cost(r.input * 1000, r.output * 1000) for r in per_call.itertuples()]
per_call.round(4)

Zero-shot accuracy: 0.88
Few-shot accuracy:  0.95
Invalid answers: 0


,input,output,"cost per 1,000 ($)"
step,,,
few-shot,606.8500,6.4500,0.1294
zero-shot,78.6277,25.9415,0.0482


**Checkpoint:** few-shot accuracy clearly higher than zero-shot (0.97 against 0.88 in the reference solution). Each few-shot call sends about 600 input tokens instead of about 100, so classifying 1,000 messages costs about 13 cents instead of about 3. The output is the same few tokens in both cases.

**Question:** compare with yesterday's lab, where zero-shot DeBERTa scored about 0.76, semantic search about 0.92 and the fine-tuned DistilBERT about 0.94 on the same ten intents. When would you use the LLM, and when the fine-tuned model? Think about accuracy, cost, speed, labels and privacy.

I would use the LLM to start, when there are few or no labels, for a prototype, for low message volumes, or when intents change often. It can also label data to train a smaller model later. I would use the fine-tuned model in production at high volume, when speed and cost per message matter and when customer data must not leave the bank. Its accuracy is close enough that these advantages outweigh the small gap.

## Challenge 4: structured output

The routing system needs more than the intent: it needs to know how urgent a message is and whether a person must handle it, in a format it can read without surprises.

1. Write a Pydantic class `Ticket` with four fields: `intent` (a `Literal` of the ten intents), `urgency` (`Literal["low", "medium", "high"]`), `needs_human` (`bool`, with a `Field` description saying it is true when the customer reports fraud, a lost or stolen card, or is very upset) and `summary` (`str`, at most ten words).
2. Write `extract(text)` with `client.chat.completions.parse`, the few-shot messages of Challenge 3 as context, an extra system message asking for the ticket, and `response_format=Ticket`. Track it as `"structured"` and return the parsed `Ticket`.
3. Extract a ticket for each of the 60 messages, and build a DataFrame with the message, the true intent and the four fields.
4. Compute the accuracy of `intent`, and show how many tickets need a person, and the urgency counts.

In [15]:
# 1. Ticket
class Ticket(BaseModel):
    intent: Literal[tuple(intents)] = Field(description="The intent of the customer's message")
    urgency: Literal["low", "medium", "high"]
    needs_human: bool = Field(description="True when customer reports fraud, a lost or stolen card, or is very upset")
    summary: str = Field(description="The customer's problem in at most ten words")

# 2. Extraction
ticket_instruction = (
    """
        Turn the customer's message into a support ticket with the following structure:
        the ticket intent, ticket urgency , whether it must be handled by a person and a short summary.
    """
)
def extract(text):
    response = track(client.chat.completions.parse(
        model = MODEL,
        temperature = 0,
        messages = few_shot_messages + [
            {'role': 'system', 'content': ticket_instruction},
            {'role': 'user', 'content': text}
        ],
        response_format=Ticket
    ),
    'structured'
    )
    return response.choices[0].message.parsed

# 3. Tickets for the 60 messages
rows = []
for row in messages.itertuples():
    ticket = extract(row.text)
    rows.append({'text': row.text, 'category': row.category, **ticket.model_dump()})

tickets = pd.DataFrame(rows)
display(tickets.head())

# 4. Accuracy, people needed, urgency
print(f'Intent accuracy: {(tickets['intent'] == tickets['category']).mean():.2f}')
print(f'Number of tickets that need a person: {tickets['needs_human'].sum()}')
print(f'Ticket urgency count: {tickets['urgency'].value_counts()}')

,text,category,intent,urgency,needs_human,summary
0,My card is being used online to make a bunch of purchases! Someone else is using it fraudulently...,compromised_card,compromised_card,high,True,Card used fraudulently for online purchases; freeze needed
1,How do I find the exchange rate?,exchange_rate,exchange_rate,low,False,Customer asks how to find the exchange rate
2,The app reverted what I topped up.,top_up_reverted,top_up_reverted,medium,False,App reverted my top up
3,What do I do to stop unauthorized transactions on my card? I was never in the place from which t...,compromised_card,compromised_card,high,True,Stop unauthorized transactions; possible card fraud
4,How long does it take to get my card? Can I choose when to receive it?,card_delivery_estimate,card_delivery_estimate,medium,False,Asks delivery time and whether delivery timing can be chosen


Intent accuracy: 0.95
Number of tickets that need a person: 14
Ticket urgency count: urgency
medium    28
high      17
low       15
Name: count, dtype: int64


**Checkpoint:** an intent accuracy close to the few-shot one (0.95 in the reference solution), every field filled with an allowed value, and about a dozen tickets (13 of 60 in the reference) that need a person, mostly fraud and lost cards.

In [16]:
# 5. Which intents does the model send to a person most often? Show the share of needs_human per true intent.
tickets.groupby('category')['needs_human'].mean().sort_values(ascending=False).round(2)

category
lost_or_stolen_card       1.00
compromised_card          0.83
top_up_reverted           0.33
card_not_working          0.17
card_arrival              0.00
card_delivery_estimate    0.00
change_pin                0.00
exchange_rate             0.00
pin_blocked               0.00
top_up_failed             0.00
Name: needs_human, dtype: float64

In [19]:
tickets.groupby("category")[["needs_human"]].mean().round(2)

,needs_human
category,
card_arrival,0.00
card_delivery_estimate,0.00
card_not_working,0.17
change_pin,0.00
compromised_card,0.83
exchange_rate,0.00
lost_or_stolen_card,1.00
pin_blocked,0.00
top_up_failed,0.00


In [20]:
pd.crosstab(tickets["category"], tickets["urgency"])

urgency,high,low,medium
category,,,
card_arrival,0,1,5
card_delivery_estimate,0,3,3
card_not_working,3,0,3
change_pin,0,5,1
compromised_card,5,0,1
exchange_rate,0,6,0
lost_or_stolen_card,6,0,0
pin_blocked,1,0,5
top_up_failed,0,0,6


**Question:** does `needs_human` follow the rule you gave it? Read a few tickets: would you trust the `urgency` and `needs_human` fields enough to route messages automatically? What would you check first?

Mostly yes, needs_human is flagged mainly for lost or stolen cards, as the rule says, but whether fraud is always caught and when a customer counts as “very upset” are left to the model’s judgement and urgency has no definition at all. I wouldn’t route automatically yet. I would first check that every compromised_card and lost_or_stolen_card message is flagged, since a missed case is the costly error, and set needs_human for those intents in code rather than leaving it to the model.

## Challenge 5: draft replies in a conversation

Finally, the assistant drafts replies for the agents. It must use the bank's real policies, and remember what the customer already said.

1. Write a system prompt with the role, the constraints (at most three sentences, never ask for the full card number, the PIN or passwords) and this context:
   - New cards arrive in 5 to 7 working days. After 10 working days, the customer can order a free replacement in the app under Card > Replace.
   - A lost or stolen card must be frozen immediately in the app under Card > Freeze; the bank then sends a new card for free.
   - A blocked PIN can be unblocked in the app under Card > PIN after confirming identity.
2. Write `reply(history, text)`: add the customer's message to `history`, call the model (tracked as `"conversation"`), add the answer as an `assistant` message, and return the answer.
3. Run this three-turn conversation and print each reply:
   - *Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.*
   - *Can I still use my old card in the meantime?*
   - *Thanks. Remind me, how many days did I say I've been waiting?*

In [22]:
# 1. System prompt
system_prompt_support = (
    """
        You are a support assitant for a bank. 
        You write a reply to the customer's message.
        You reply with at most 3 sentences and you should never ask a customer for 
        the full card number, the PIN or the password.
        Facts to know:
        - All new card arrive within 5 to 7 working days.
        - After 10 working days the customer can order a free replacement in the app,
        which can be found under Card -> Replace.
        - If a card is stolen or lost it must be immediately frozen in the app, 
        which can be found under Card -> Freeze. Afterward, the bank will send the customer
        a new card for free.
        - If a PIN is blocked, it can be unblocked through the app, under Card -> PIN,
        after the identity has been confirmed.
    """
)

# 2. Reply with history
def reply(history, text):
    history.append({'role': 'user', 'content': text})
    response = track(
        client.chat.completions.create(
            model = MODEL,
            messages=history
        ),
        'conversation'
    )
    answer = response.choices[0].message.content
    history.append({'role': 'assistant', 'content': answer})
    return answer

# 3. Three turns
system_message = [{'role': 'system', 'content': system_prompt_support}]

conversation = [
    """
    Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.
    Can I still use my old card in the meantime?
    Thanks. Remind me, how many days did I say I've been waiting?
    """
]

for text in conversation:
    print(f'Customer: {text}')
    print(f'Assitant: {reply(system_message, text)}')
    print('---')

print(f'Messages sent in the last call: {len(conversation) - 1}')


Customer: 
    Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.
    Can I still use my old card in the meantime?
    Thanks. Remind me, how many days did I say I've been waiting?
    
Assitant: Hi Marta—if you ordered your new card 12 working days ago, it should be followed up now. Yes, you can usually keep using your old card until the replacement arrives, but if it’s been deactivated you’ll need the new one. You can also order a free replacement in the app under **Card → Replace** after 10 working days.
---
Messages sent in the last call: 0


**Checkpoint:** three short replies. The first one uses the policy (more than 10 working days, so a free replacement under Card > Replace) and the customer's name. The third proves the history works: it remembers the 12 working days. No reply asks for the PIN or the card number.

**Question:** the second question (can I use my old card?) is not covered by the policies you gave. What did the assistant answer, and why is that a risk for the bank? How would you reduce it?

Use of the old card is not covered in the rules given to the agent, which means that it inveted a new rule by itself. This is risky for a bank because the customer will trust a made-up rule, which can lead to failed payments or further customer complaints. To migitage this, we can add another rule to the set above "Only use these rules and if a customer asks a question for which you don't have the answer, redirect them to a person."

## What did it cost?

Run the cell below to see the tokens and the cost of every step of the lab.

In [23]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

Total: 431 calls, $0.0564


,input,output,calls,cost ($)
step,,,,
first call,113,132,2,0.00019
zero-shot,14782,4877,188,0.00905
few-shot,36411,387,60,0.00777
structured,156771,6335,180,0.03927
conversation,241,76,1,0.00014


**Checkpoint:** about 180 calls and a total of about 2 cents (well under 5). The structured step is the most expensive, because it sends the 20 examples with every call; the zero-shot classifier is the cheapest.

## Extra: let the assistant check the delivery status

Agents can look up where a customer's card is. Give the assistant the same ability with **function calling**.

1. Write `get_card_status(customer_id)` that looks the customer up in the dictionary `CARD_ORDERS` below and returns a dictionary with the status, or an error if the customer is unknown.
2. Describe it in a `tools` list (one required string parameter, `customer_id`, with `"strict": True`).
3. Run a conversation with the support prompt where the customer writes *My customer id is C-1043, where is my new card?*: when the model asks for the tool, run it, send the result back with the role `tool`, and print the final answer. Track every call as `"extra"`.
4. Try it with the unknown customer `C-9999`. Does the assistant handle the error well?

In [ ]:
CARD_ORDERS = {
    "C-1043": {"ordered": "2026-10-12", "status": "shipped", "carrier": "Correos", "expected": "2026-10-23"},
    "C-2210": {"ordered": "2026-10-20", "status": "being printed", "carrier": None, "expected": "2026-10-29"},
}

# Your code here

**Conclusions:** *Your answer here.*